# 第 4 章 · 委派、容错与人工介入

> 对应文档：**Agents → Configure the harness → Planning and delegation / Name your agent / Fault tolerance / Guardrails / Steering**
> https://docs.langchain.com/oss/python/langchain/agents#planning-and-delegation
>
> 每个小节的结构都一样：**文档怎么说 → 平台上对应哪里 → 练习怎么做 → 看什么算证据**。
> Python 块是文档里的原写法，**只作对照，不用运行** —— 在这个平台上，同样的东西是
> 在 agent 编辑器的五个页签里填字段：Overview / Prompt / Tools / Skills / JSON。


## 1 · Planning and delegation（todo + 子代理）

> Complex tasks often exceed what one context window can handle. Delegation lets the main
> agent break work into pieces, hand them to subagents that each run in their own isolated
> context, and stay focused on coordination rather than execution. Work can run in parallel;
> the main agent's context stays clean.

```python
from deepagents.middleware import SubagentMiddleware
agent = create_agent(model=..., tools=[...], middleware=[SubagentMiddleware(...)])
```

**平台上对应哪里**

| 能力 | 位置 |
|---|---|
| todo 规划 | 勾 `write_todos` / `read_todos` 两个工具 |
| 子代理 | JSON 页签的 `subagents`（`name` + `description`，按描述路由） |
| 观察 | 回复里的子代理活动、Thinking 里的步骤数 |

**练习**

- **A 自己做**：问一个多步任务（`查这三家公司的最近动态，汇总成表`），看它的步骤数和上下文。
- **B 委派**：同一个问题，加上子代理配置，再看。

**看什么算证据**

- 委派带来的**唯一**好处是**上下文隔离**：主 agent 的窗口不被三份原始材料塞满。
- 所以判据是：主对话的步骤/内容是否更短，而**答案是否依然完整**。
- **代价也要讲**：一个单步任务也去委派，是纯开销 —— 一次工具调用变成一整轮 agent 循环。

> 提醒：子代理看不到主 agent 的其它上下文。它会独立回答，所以"用中文""别用列表"
> 这类约束必须在子代理的 `description`/prompt 里再说一遍，否则它不知道。

## 2 · Name your agent

> Optionally use an identifier for the agent. Especially useful when embedding the agent as a
> subgraph in multi-agent systems.

**平台上对应哪里**：agent 的名称/描述（列表里显示、子代理路由时用）。

**为什么它不只是个名字**：子代理**按描述被选中**。描述写得含糊，主 agent 就不会委派，
或者委派给错的。这和技能触发是同一个机制 —— **描述是路由依据，不是说明文字**。

## 3 · Fault tolerance（重试、回退、上限）

> Agents in production encounter failures that rarely appear in development: rate limits,
> model timeouts, transient API errors. Fault tolerance middleware handles these at the
> infrastructure level so your tools and business logic don't need try/catch around every call.

**平台上对应哪里**

| 能力 | 位置 | 现状 |
|---|---|---|
| 工具重试 | JSON 的 `middleware` 里 `tool_retry` | 可配，无 UI |
| 模型回退 | JSON 的 `fallback_models` | 可配，无 UI |
| 调用上限 | Overview 的 recursion limit | 可配 |
| 实际手感 | 模型被限流/超时时 | 会看到报错，**不会自动换模型**（除非配了 fallback） |

**练习**：故意把一个**不存在的模型名**填进 Model，问一句。看报错是"模型不可用"还是
"答案为空" —— 这决定了你在生产里能不能定位问题。

> 这一节要教的判断：**重试/回退属于基础设施，不属于 prompt。** 把它写进 prompt
> （"如果失败就重试"）没有意义，因为它失败的时候已经不在了。

## 4 · Guardrails（确定性策略）

> Some policies can't live in a prompt — they need to be enforced deterministically regardless
> of what the model does. Guardrails intercept data as it flows through the agent loop.

**平台上对应哪里**：Overview 的 token 上限（`max_total_tokens` / 淘汰阈值 / 摘要阈值）。

**这一节最重要的一句话**：**能写进 prompt 的政策都不是政策，是建议。**
prompt 可以被绕过（换个问法、让它"只是举例"），而确定性拦截不能。

**平台缺口**：文档举的例子是 `PIIMiddleware`（脱敏），这一版**没有实现**。
所以"不要把身份证号写进日志"这种要求，现在只能靠 prompt —— 也就是靠自觉。

## 5 · Steering（人在关键点介入）

> Full autonomy isn't always appropriate. Steering lets you place humans at specific decision
> points — before destructive writes, expensive API calls, or anything requiring judgment —
> without restructuring your agent. The agent pauses and waits; a human approves, edits, or
> rejects; execution continues.

```python
from langchain.agents.middleware import HumanInTheLoopMiddleware
agent = create_agent(model=..., tools=[...], middleware=[HumanInTheLoopMiddleware(...)])
```

**平台上对应哪里**：聊天面板的设置菜单里开启 HITL（`MiddlewareType.HITL`），以及审批接口
`/api/hitl/`。触发点是**工具调用之前**。

**练习**：给 agent 一个会写文件的工具，开启 HITL，然后让它"把结果写进 generated/"。
这时应该**停下来等你批准**，而不是直接写。

**看什么算证据**

- 有没有**暂停**，以及暂停的位置是不是"写之前"。
- 拒绝之后，agent 的行为是否合理（重新给方案，还是硬写）。
- 这一节和"prompt 里写：重要操作前问我"**不是一回事**：前者拦得住，后者只是礼貌。

## 小结：七块能力，怎么判断该用哪块

| 你想要 | 用哪个 | 判据 |
|---|---|---|
| 让它能做事 | Tools / Execution environment | 有没有工具调用 + 有没有文件 |
| 让它按你的方式说话 | System prompt / `AGENTS.md` | 输出形状 |
| 让它在窗口里待得久 | Summarization / Skills | 长任务不糊 |
| 让它拆活 | Planning & delegation | 主上下文是否更干净 |
| 让它别乱写 | Steering (HITL) | 是否停下来等批准 |
| 让它扛住限流 | Fault tolerance | 报错是否可定位 |
| 让它不能违规 | Guardrails | 拦截是否确定（prompt 做不到） |

**一句话**：prompt 决定**它怎么表现**，harness 决定**它能不能、该不该**。